# 01 · Items: every core PPD dataflow

This notebook builds the **columns of the household matrix**: every combination of the five core fields of a PPD
dataflow. Everything is in this notebook; it reads only the PPD drafts' term lists (typed in below) and the researched
device flows.

**Sources:** draft-dsmullen-ppd-taxonomy-05 (§6: core terms, §6.7: qualifiers) and draft-dsmullen-ppd-protocol-04
(§7.11–7.12: policy rules and constraints), both in the capstone's `docs/ietf-drafts/`.

**Output:** `data/items.csv` (6,272 rows) and `data/corpus/flows.csv` (the real device flows).

In [1]:
import itertools, json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
APP = ROOT.parent / "ppd-preference-app"   # optional: the app repository, for refreshing the device flows
(DATA / "corpus").mkdir(parents=True, exist_ok=True)

## 1. The five core fields (taxonomy-05 §6)

A PPD dataflow is described by five fields, each taking exactly one **core term** (prefix `ppd:`). These are the term
lists of taxonomy-05, copied verbatim:

| Field | Section | Meaning |
| --- | --- | --- |
| `data_type` | §6.2 | *what* data |
| `purpose` | §6.3 | *why* it is handled |
| `action` | §6.4 | *how*: collection, use, transfer, inference (a flat list, no vendor refinements) |
| `source` | §6.5 | *where it comes from* (its immediate origin) |
| `handling_context` | §6.6 | *where it goes / happens*: the household, the vendor, a third party, the public |

In [2]:
DATA_TYPES = ["ppd:identifierData", "ppd:profileData", "ppd:sensorData", "ppd:contentData",
              "ppd:locationData", "ppd:deviceAndNetworkData", "ppd:usageData"]                      # §6.2
PURPOSES = ["ppd:coreFunctionality", "ppd:personalization", "ppd:communicationsAndNotifications", "ppd:security",
            "ppd:diagnosticsAndMaintenance", "ppd:analyticsAndImprovement", "ppd:legalCompliance",
            "ppd:advertisingAndMarketing"]                                                           # §6.3
ACTIONS = ["ppd:collection", "ppd:use", "ppd:transfer", "ppd:inference"]                             # §6.4
SOURCES = ["ppd:userProvided", "ppd:participantObserved", "ppd:participantGenerated", "ppd:householdProvided",
           "ppd:vendorProvided", "ppd:thirdPartyProvided", "ppd:derivedFromPriorData"]               # §6.5
HANDLING_CONTEXTS = ["ppd:householdContext", "ppd:vendorContext", "ppd:thirdPartyContext", "ppd:publicAudience"]  # §6.6

FIELDS = ["data_type", "purpose", "action", "source", "handling_context"]
FIELD_TERMS = dict(zip(FIELDS, [DATA_TYPES, PURPOSES, ACTIONS, SOURCES, HANDLING_CONTEXTS]))
short = lambda term: term.split(":", 1)[1]
pd.Series({f: len(t) for f, t in FIELD_TERMS.items()}, name="core terms")

data_type           7
purpose             8
action              4
source              7
handling_context    4
Name: core terms, dtype: int64

## 2. Why the matrix has exactly these columns

**One column per combination of the five fields: 7 × 8 × 4 × 7 × 4 = 6,272.**

This is the granularity of a PPD household policy itself (protocol-04 §7.11):
- a policy rule names **one** value of each of the five core fields, plus an effect (`allow` or `deny`);
- a policy *"SHOULD NOT contain two Policy Rule Objects with the same core dimensions but different effect values"*.

So a household policy holds at most one rule per combination, and one decision per combination is exactly what a
policy can express.

In [3]:
rows = list(itertools.product(DATA_TYPES, PURPOSES, ACTIONS, SOURCES, HANDLING_CONTEXTS))
items = pd.DataFrame(rows, columns=FIELDS)
items.insert(0, "item_id", range(len(items)))
print(f"{len(items):,} items")
items.head()

6,272 items


,item_id,data_type,purpose,action,source,handling_context
0,0,ppd:identifierData,ppd:coreFunctionality,ppd:collection,ppd:userProvided,ppd:householdContext
1,1,ppd:identifierData,ppd:coreFunctionality,ppd:collection,ppd:userProvided,ppd:vendorContext
2,2,ppd:identifierData,ppd:coreFunctionality,ppd:collection,ppd:userProvided,ppd:thirdPartyContext
3,3,ppd:identifierData,ppd:coreFunctionality,ppd:collection,ppd:userProvided,ppd:publicAudience
4,4,ppd:identifierData,ppd:coreFunctionality,ppd:collection,ppd:participantObserved,ppd:householdContext


## 3. Qualifiers are conditions inside a decision, not columns (taxonomy-05 §6.7)

A flow can carry optional **qualifiers** in its `constraints`: retention, processing boundary, jurisdiction. They do
not identify the flow; they *refine* a rule: *"structured qualifiers that refine the rule"* (protocol-04 §7.11–7.12).

So in the matrix a qualifier appears **inside the household's answer** for an item, as a condition:
`allow_if: retention=ppd:ephemeral` is one PPD rule, `effect: allow` with `constraints: {retention: ppd:ephemeral}`.

Which qualifier values can make a flow acceptable, and where they apply:

| Qualifier | Value used as a condition | Applies to | Draft |
| --- | --- | --- | --- |
| retention | `ppd:ephemeral` ("not intended to result in durable persistence") | every action; for transfer it means the receiver doesn't keep it | §6.7.1 |
| processing boundary | `ppd:inHomeOnly`, `ppd:onDeviceOnly` | **only use and inference**: "processing-placement qualifier for use or inference" (protocol §7.12); a qualifier outside its defined applicability is invalid (§6.7) | §6.7.2 |
| jurisdiction | (scope + country codes) | not modelled yet | §6.7.3 |

`ppd:indefinite` and `ppd:approvedRemoteProcessing` never make a flow *more* acceptable, so they are not conditions.
In-home / on-device processing narrows where processing runs **within** the handling context; it does not move the
flow home (§6.6, §6.7.2).

In [4]:
LIMIT_EPHEMERAL = "retention=ppd:ephemeral"
LIMIT_IN_HOME = "processing_boundary=ppd:inHomeOnly"
LIMIT_ON_DEVICE = "processing_boundary=ppd:onDeviceOnly"
BOUNDARY_ACTIONS = {"ppd:use", "ppd:inference"}

def limits_for(action):
    return [LIMIT_EPHEMERAL] + ([LIMIT_IN_HOME, LIMIT_ON_DEVICE] if action in BOUNDARY_ACTIONS else [])

items["limits"] = items.action.map(lambda a: "|".join(limits_for(a)))
items["leaves_home"] = items.handling_context != "ppd:householdContext"   # same test as the app's leavesHome
items.groupby("action").limits.first()

action
ppd:collection                              retention=ppd:ephemeral
ppd:inference     retention=ppd:ephemeral|processing_boundary=pp...
ppd:transfer                                retention=ppd:ephemeral
ppd:use           retention=ppd:ephemeral|processing_boundary=pp...
Name: limits, dtype: str

## 4. Plausibility flag (consistency with the term definitions)

Only combinations that the definitions themselves rule out are flagged; nothing is removed.

- `ppd:publicAudience` is *"a disclosure context in which data is made available to the public"* (§6.6).
  Disclosing is a **transfer** (§6.4: "making data available to a different recipient or handling context").
  Collection brings data into the participant's own context, use operates "without disclosing it", inference derives
  data. So public audience fits only `transfer`.

In [5]:
items["implausible_reason"] = ""
public_not_transfer = (items.handling_context == "ppd:publicAudience") & (items.action != "ppd:transfer")
items.loc[public_not_transfer, "implausible_reason"] = "public audience is a disclosure context; only transfer discloses"
items["plausible"] = items.implausible_reason == ""
print(f"plausible {items.plausible.sum():,}, flagged {(~items.plausible).sum():,}")

plausible 5,096, flagged 1,176


## 5. The real devices (the corpus)

Our team's research workbook describes what four real products do with data; the app's device library was built from
it. Each real flow is already written in core terms. If the app repository is next to this one, the snapshot is
refreshed from it; otherwise the committed `data/corpus/flows.csv` is used.

In [6]:
def import_corpus(app: Path) -> pd.DataFrame:
    mock = app / "public" / "mock"
    rows = []
    for entry in json.loads((mock / "library.json").read_text())["devices"]:
        device = json.loads((mock / "devices" / f"{entry['id']}.json").read_text())
        for flow in device["flows"]:
            t, c = flow["taxonomy"], flow["taxonomy"].get("constraints") or {}
            rows.append({"device_id": device["id"], "device_name": device["name"], "flow_id": flow["id"],
                         "title": flow["title"], "phase": flow.get("phase", ""), "evidence": flow.get("evidence", ""),
                         **{k: t[k] for k in FIELDS},
                         "retention": c.get("retention", ""), "processing_boundary": c.get("processing_boundary", "")})
    return pd.DataFrame(rows)

corpus_path = DATA / "corpus" / "flows.csv"
if (APP / "public" / "mock" / "library.json").exists():
    corpus = import_corpus(APP)
    corpus.to_csv(corpus_path, index=False)
    print("refreshed from", APP.name)
else:
    print("using the committed snapshot")
corpus = pd.read_csv(corpus_path)
corpus.groupby("device_name").agg(flows=("flow_id", "size"))

refreshed from ppd-preference-app


,flows
device_name,
Aqara Thermostat Hub W200,22
LG OLED65C6PUA C6,19
Level Lock,21
Reolink E1 Zoom,26


Many different real flows land on the **same** core combination (the taxonomy is deliberately coarse). For example
"Wi-Fi password entered at setup" on all four devices is one item. So 88 real flows cover only **52 items**.

In [7]:
grouped = corpus.groupby(FIELDS).agg(corpus_count=("flow_id", "size"),
                                     corpus_devices=("device_id", lambda s: "|".join(sorted(set(s))))).reset_index()
items = items.merge(grouped, on=FIELDS, how="left")
items["corpus_count"] = items.corpus_count.fillna(0).astype(int)
items["corpus_devices"] = items.corpus_devices.fillna("")
items["in_corpus"] = items.corpus_count > 0
print(f"{len(corpus)} real flows -> {items.in_corpus.sum()} distinct items")
items[items.in_corpus].sort_values("corpus_count", ascending=False).head(8)[FIELDS + ["corpus_count", "corpus_devices"]]

88 real flows -> 52 distinct items


,data_type,purpose,action,source,handling_context,corpus_count,corpus_devices
1,ppd:identifierData,ppd:coreFunctionality,ppd:collection,ppd:userProvided,ppd:vendorContext,6,aqara-w200|level-lock|lg-oled65c6pua-c6|reolin...
4480,ppd:deviceAndNetworkData,ppd:coreFunctionality,ppd:collection,ppd:userProvided,ppd:householdContext,6,aqara-w200|level-lock|lg-oled65c6pua-c6|reolin...
4993,ppd:deviceAndNetworkData,ppd:diagnosticsAndMaintenance,ppd:transfer,ppd:participantGenerated,ppd:vendorContext,5,aqara-w200|level-lock|lg-oled65c6pua-c6
2236,ppd:sensorData,ppd:security,ppd:inference,ppd:derivedFromPriorData,ppd:householdContext,4,reolink-e1-zoom
2753,ppd:contentData,ppd:coreFunctionality,ppd:transfer,ppd:participantGenerated,ppd:vendorContext,3,reolink-e1-zoom
6001,ppd:usageData,ppd:analyticsAndImprovement,ppd:transfer,ppd:participantGenerated,ppd:vendorContext,3,aqara-w200|lg-oled65c6pua-c6|reolink-e1-zoom
1796,ppd:sensorData,ppd:coreFunctionality,ppd:collection,ppd:participantObserved,ppd:householdContext,3,aqara-w200|lg-oled65c6pua-c6|reolink-e1-zoom
3028,ppd:contentData,ppd:security,ppd:collection,ppd:participantObserved,ppd:householdContext,2,reolink-e1-zoom


In [8]:
# Which core terms the real devices use, per field (and which they never use).
for f in FIELDS:
    used = corpus[f].value_counts()
    never = [short(t) for t in FIELD_TERMS[f] if t not in used.index]
    print(f"{f:17} " + ", ".join(f"{short(k)} {v}" for k, v in used.items()) + (f"   | never: {', '.join(never)}" if never else ""))

data_type         deviceAndNetworkData 25, contentData 19, sensorData 14, identifierData 11, usageData 10, locationData 6, profileData 3
purpose           coreFunctionality 51, security 11, diagnosticsAndMaintenance 8, advertisingAndMarketing 7, analyticsAndImprovement 7, personalization 2, communicationsAndNotifications 2   | never: legalCompliance
action            collection 36, transfer 31, use 11, inference 10
source            participantGenerated 24, userProvided 23, derivedFromPriorData 18, participantObserved 13, householdProvided 4, vendorProvided 4, thirdPartyProvided 2
handling_context  vendorContext 43, householdContext 38, thirdPartyContext 7   | never: publicAudience


## 6. Checks

These must hold; the notebook stops if one fails.

In [9]:
assert len(items) == 7 * 8 * 4 * 7 * 4 == 6272
assert items.in_corpus.sum() == corpus[FIELDS].drop_duplicates().shape[0]
assert not (items.in_corpus & ~items.plausible).any(), "a real device declares an item flagged implausible"
assert set(items[items.limits.str.contains("processing_boundary")].action) == BOUNDARY_ACTIONS
assert set(items[(items.handling_context == "ppd:publicAudience") & items.plausible].action) == {"ppd:transfer"}
pd.DataFrame({"count": {
    "items": len(items), "leave the home": items.leaves_home.sum(),
    "allow a processing-boundary condition": items.limits.str.contains("processing_boundary").sum(),
    "plausible": items.plausible.sum(), "declared by real devices": items.in_corpus.sum(),
    "real but flagged implausible": (items.in_corpus & ~items.plausible).sum()}})

,count
items,6272
leave the home,4704
allow a processing-boundary condition,3136
plausible,5096
declared by real devices,52
real but flagged implausible,0


## 7. Save

In [10]:
cols = ["item_id", *FIELDS, "leaves_home", "limits", "implausible_reason", "plausible", "corpus_count", "corpus_devices", "in_corpus"]
items[cols].to_csv(DATA / "items.csv", index=False)
print("wrote", (DATA / "items.csv").relative_to(ROOT))

wrote data/items.csv
